# Kapitola 7: Vytváranie chatovacích aplikácií
## Rýchly štart API Microsoft Foundry Models

Tento zápisník je upravený z [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), ktorý obsahuje zápisníky pristupujúce k službám [Azure OpenAI](aoai-assignment.ipynb).

> **Poznámka:** GitHub Models bude ukončený koncom júla 2026. Tento zápisník teraz cieli na [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), ktorý ponúka rovnaký katalóg modelov na bezplatné vyskúšanie a skúsenosti so SDK Azure AI Inference.


# Prehľad  
„Veľké jazykové modely sú funkcie, ktoré mapujú text na text. Vzhľadom na vstupný reťazec textu sa veľký jazykový model snaží predpovedať text, ktorý príde ďalej“(1). Tento „rýchly úvod“ notebook predstaví používateľom základné koncepty LLM, základné požiadavky balíka na začiatok práce s AML, mierny úvod do návrhu podnetov a niekoľko krátkych príkladov rôznych prípadov použitia. 


## Obsah  

[Prehľad](#overview)  
[Ako používať OpenAI Service](#how-to-use-openai-service)  
[1. Vytvorenie vášho OpenAI Service](#1.-creating-your-openai-service)  
[2. Inštalácia](#2.-installation)    
[3. Prístupové údaje](#3.-credentials)  

[Použitia](#use-cases)    
[1. Zhrnutie textu](#1.-summarize-text)  
[2. Klasifikácia textu](#2.-classify-text)  
[3. Generovanie nových názvov produktov](#3.-generate-new-product-names)  
[4. Doladenie klasifikátora](#4.fine-tune-a-classifier)  

[Referencie](#references)


### Vytvorte si svoj prvý prompt  
Tento krátky cvičenie poskytne základný úvod do posielania promptov modelu v Microsoft Foundry Models pre jednoduchý úloh "zhrnutie".


**Kroky**:  
1. Nainštalujte si knižnicu `azure-ai-inference` vo vašom python prostredí, ak ste to ešte nespravili.  
2. Načítajte štandardné pomocné knižnice a nastavte si poverenia Microsoft Foundry Models.  
3. Vyberte si model pre váš úloh  
4. Vytvorte jednoduchý prompt pre model  
5. Odoslať svoju požiadavku na modelové API!


### 1. Inštalácia `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Importujte pomocné knižnice a vytvorte poverenia


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Nájdenie správneho modelu  
Modely ako GPT-4o a GPT-4o mini dokážu rozumieť a generovať prirodzený jazyk a sú dostupné v katalógu Microsoft Foundry Models spolu s modelmi od Meta, Mistral, Cohere a Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-5-mini"


## 4. Návrh promptov  

"Kúzlo veľkých jazykových modelov spočíva v tom, že keď sú trénované na minimalizáciu tejto chyby predpovede na obrovskom množstve textu, modely sa nakoniec naučia koncepty užitočné pre tieto predpovede. Napríklad sa naučia koncepty ako"(1):

* ako sa píše pravopisne správne
* ako funguje gramatika
* ako parafrázovať
* ako odpovedať na otázky
* ako viesť rozhovor
* ako písať v mnohých jazykoch
* ako programovať
* atď.

#### Ako ovládať veľký jazykový model  
"Zo všetkých vstupov do veľkého jazykového modelu je bezpochyby najvplyvnejším textový prompt(1).

Veľké jazykové modely je možné promptovať, aby produkovali výstup niekoľkými spôsobmi:

Inštrukcia: Povedzte modelu, čo chcete
Dokončenie: Podnecujte model, aby dokončil začiatok toho, čo chcete
Demonštrácia: Ukážte modelu, čo chcete, buď:
Niekoľko príkladov v texte promptu
Niekoľko stoviek alebo tisícok príkladov v tréningovej dátovej sade pre doladenie"



#### Existujú tri základné pravidlá pre tvorbu promptov:

**Ukáž a povedz**. Jasne dajte najavo, čo chcete, buď cez inštrukcie, príklady, alebo kombináciu oboch. Ak chcete, aby model zoradil zoznam položiek v abecednom poradí alebo klasifikoval odsek podľa sentimentu, ukážte mu, že to chcete.

**Poskytnite kvalitné dáta**. Ak sa snažíte vytvoriť klasifikátor alebo chcete, aby model nasledoval vzor, uistite sa, že máte dostatok príkladov. Nezabudnite si skontrolovať svoje príklady — model je zvyčajne dosť inteligentný na to, aby cez základné pravopisné chyby videl a poskytol odpoveď, ale môže tiež predpokladať, že je to zámerné, čo môže ovplyvniť odpoveď.

**Skontrolujte svoje nastavenia.** Nastavenia teploty a top_p riadia, aký je model deterministický pri generovaní odpovede. Ak žiadate odpoveď, kde je len jedna správna možnosť, chcete ich nastaviť nižšie. Ak hľadáte rozmanitejšie odpovede, môžete ich nastaviť vyššie. Najčastejšou chybou ľudí pri týchto nastaveniach je, že si myslia, že sú to ovládače "bystrosti" alebo "kreativity".


Zdroj: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Odošli! 


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Opakujte ten istý hovor, ako sa výsledky porovnávajú?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Zhrnúť text  
#### Výzva  
Zhrňte text tak, že na koniec textového úryvku pridáte 'tl;dr:'. Všímajte si, ako model rozumie, ako vykonávať viaceré úlohy bez ďalších inštrukcií. Môžete experimentovať s popisnejšími výzvami než tl;dr, aby ste upravili správanie modelu a prispôsobili zhrnutie, ktoré dostanete(3).  

Nedávna práca preukázala značné zlepšenia v mnohých NLP úlohách a benchmarkoch vďaka predškoleniu na veľkom korpuse textu a následnému doladeniu na konkrétnu úlohu. Hoci je táto metóda spravidla architektúrou nezávislá od úloh, stále vyžaduje doladenie pre konkrétnu úlohu na tisícoch alebo desaťtisícoch príkladov. Naopak, ľudia zvyčajne dokážu vykonať novú jazykovú úlohu už z niekoľkých príkladov alebo jednoduchých inštrukcií - čo súčasné NLP systémy stále vo veľkej miere nezvládajú. Tu ukazujeme, že zväčšenie jazykových modelov výrazne zlepšuje výkonnosť v úlohách nezávislých od úlohy s niekoľkými príkladmi, niekedy dokonca dosahujúc konkurencieschopnosť s predchádzajúcimi štandardmi v prístupe doladenia.  



Tl;dr


# Cvičenia pre niekoľko prípadov použitia  
1. Zhrnúť text  
2. Klasifikovať text  
3. Generovať nové názvy produktov


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Triediť text  
#### Výzva  
Triediť položky do kategórií poskytnutých v čase inferencie. V nasledujúcom príklade uvádzame kategórie aj text na klasifikáciu v podnete (*playground_reference). 

Zákaznícka dopyt: Dobrý deň, nedávno sa mi pokazilo jedno tlačidlo na klávesnici notebooku a budem potrebovať náhradu:

Zaradená kategória:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generovanie nových názvov produktov
#### Výzva
Vytvorte názvy produktov z príkladových slov. V promptu uvádzame informácie o produkte, pre ktorý budeme generovať názvy. Tiež poskytujeme podobný príklad, aby sme ukázali vzor, ktorý chceme získať. Nastavili sme tiež vysokú hodnotu teploty, aby sme zvýšili náhodnosť a inovatívnosť odpovedí.

Popis produktu: Domáci výrobný stroj na mliečne koktaily
Počiatočné slová: rýchly, zdravý, kompaktný.
Názvy produktov: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Popis produktu: Pár topánok, ktoré sadnú na akúkoľvek veľkosť nohy.
Počiatočné slová: prispôsobivý, vhodný, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Referencie  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najlepšie postupy pri jemnom doladení GPT modelov na klasifikáciu textu](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Pre viac pomoci  
[Tím pre komercializáciu OpenAI](AzureOpenAITeam@microsoft.com) 


# Prispievatelia
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vyhlásenie o zodpovednosti**:
Tento dokument bol preložený pomocou AI prekladateľskej služby [Co-op Translator](https://github.com/Azure/co-op-translator). Hoci sa snažíme o presnosť, vezmite prosím na vedomie, že automatické preklady môžu obsahovať chyby alebo nepresnosti. Pôvodný dokument v jeho natívnom jazyku by mal byť považovaný za autoritatívny zdroj. Pre kritické informácie sa odporúča profesionálny ľudský preklad. Nie sme zodpovední za žiadne nedorozumenia alebo nesprávne interpretácie vyplývajúce z použitia tohto prekladu.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
